BASE DE DATOS GITHUB

In [1]:
import sys
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LinearRegression # 1. EL CAMBIO: Regresión Pura (sin freno)
from sklearn.preprocessing import MinMaxScaler

# =============================================================================
# 1. CONECTAR FUNCIONES DE PROCESADO
# =============================================================================
ruta_hugo = Path("../../Hugo").resolve()
if str(ruta_hugo) not in sys.path:
    sys.path.insert(0, str(ruta_hugo))

try:
    from channel_estimator import compute_channel_matrix_from_iq_paths
    from channel_features_complete import extract_channel_matrix_features
except ImportError:
    print("❌ ERROR: No se han podido importar las funciones de Hugo.")
    exit()

# =============================================================================
# 2. FASE DE ENTRENAMIENTO (UNIVERSO 1: BBDD ANTIGUA)
# =============================================================================
print("🧠 1. Entrenando IA Lineal con la BBDD Antigua...")

df_antiguo = pd.read_csv('../../data/dataset_features_temperatura.csv')

# Las 3 variables que demostramos que son perfectas y limpias
variables_clave = [
    'doppler_variance_energy', 
    'dH_dt_mean', 
    'svd_sigma_ratio'
]

variables_finales = [v for v in variables_clave if v in df_antiguo.columns]

X_train_bruto = df_antiguo[variables_finales]
y_train = df_antiguo['temperature']

# ESCALADOR 1
scaler_train = MinMaxScaler()
X_train_scaled = scaler_train.fit_transform(X_train_bruto)

# CEREBRO LINEAL PURO (Sin aplastar los rangos)
modelo_ia = LinearRegression()
modelo_ia.fit(X_train_scaled, y_train)

print("✅ IA entrenada con éxito en el Universo Antiguo.")

# =============================================================================
# 3. EXTRACCIÓN DE DATOS EN VIVO (UNIVERSO 2: VASO CARTÓN)
# =============================================================================
print("\n📡 2. Procesando grabaciones del Vaso de Cartón...")
RUTA_YAML = Path("../../data/Modulator.yaml")
BBDD_DIR = Path("../Datos TEST Vaso carton")

pruebas = [
    {"nombre": "Vaso Frío - Muestra 1", "tx": BBDD_DIR/"frio"/"iq_tx_1.bin", "rx": BBDD_DIR/"frio"/"iq_rx_1.bin"},
    {"nombre": "Vaso Frío - Muestra 2", "tx": BBDD_DIR/"frio"/"iq_tx_2.bin", "rx": BBDD_DIR/"frio"/"iq_rx_2.bin"},
    {"nombre": "Vaso Templado - Muestra 1", "tx": BBDD_DIR/"templado"/"iq_tx_1.bin", "rx": BBDD_DIR/"templado"/"iq_rx_1.bin"},
    {"nombre": "Vaso Templado - Muestra 2", "tx": BBDD_DIR/"templado"/"iq_tx_2.bin", "rx": BBDD_DIR/"templado"/"iq_rx_2.bin"},
    {"nombre": "Vaso Caliente - Muestra 1", "tx": BBDD_DIR/"caliente"/"iq_tx_1.bin", "rx": BBDD_DIR/"caliente"/"iq_rx_1.bin"},
    {"nombre": "Vaso Caliente - Muestra 2", "tx": BBDD_DIR/"caliente"/"iq_tx_2.bin", "rx": BBDD_DIR/"caliente"/"iq_rx_2.bin"}
]

datos_test_lista = []
nombres_validos = []

for p in pruebas:
    if not p["tx"].exists() or not p["rx"].exists():
        continue
        
    try:
        H = compute_channel_matrix_from_iq_paths(
            tx_path=p["tx"], rx_path=p["rx"], yaml_path=RUTA_YAML,
            fftshift=False, normalize_fft=False, trim_to_complete_frames=True,
            output_order="mk", verbose=False
        )
        todas_features = extract_channel_matrix_features(H, input_order="mk")
        
        features_filtradas = {var: todas_features[var] for var in variables_finales}
        datos_test_lista.append(features_filtradas)
        nombres_validos.append(p["nombre"])
        
    except Exception as e:
        print(f"❌ Error en {p['nombre']}: {e}")

# =============================================================================
# 4. ADAPTACIÓN DE DOMINIO Y PREDICCIÓN
# =============================================================================
print("-" * 50)
print("🎯 RESULTADOS FINALES DE PREDICCIÓN")
print("-" * 50)

if len(datos_test_lista) > 0:
    df_test_bruto = pd.DataFrame(datos_test_lista)
    
    scaler_test = MinMaxScaler()
    
    # EL TRUCO DEL ESPEJO: Invertimos los datos nuevos (1.0 - ...) 
    # para que la BBDD Antigua crea que los entiende.
    X_test_scaled = 1.0 - scaler_test.fit_transform(df_test_bruto)
    
    predicciones = modelo_ia.predict(X_test_scaled)
    
    for nombre, temp in zip(nombres_validos, predicciones):
        print(f"🌡️ [{nombre}] -> Predicción IA: {temp:.1f} ºC")
else:
    print("❌ No se ha podido procesar ningún archivo de test.")

print("-" * 50)

🧠 1. Entrenando IA Lineal con la BBDD Antigua...
✅ IA entrenada con éxito en el Universo Antiguo.

📡 2. Procesando grabaciones del Vaso de Cartón...
--------------------------------------------------
🎯 RESULTADOS FINALES DE PREDICCIÓN
--------------------------------------------------
🌡️ [Vaso Frío - Muestra 1] -> Predicción IA: 117.0 ºC
🌡️ [Vaso Frío - Muestra 2] -> Predicción IA: 114.4 ºC
🌡️ [Vaso Templado - Muestra 1] -> Predicción IA: 109.1 ºC
🌡️ [Vaso Templado - Muestra 2] -> Predicción IA: 103.6 ºC
🌡️ [Vaso Caliente - Muestra 1] -> Predicción IA: 49.8 ºC
🌡️ [Vaso Caliente - Muestra 2] -> Predicción IA: 50.0 ºC
--------------------------------------------------
